In [1]:
import sys
from pathlib import Path

# 노트북이 data_module/ 안에 있을 때, 상위(=프로젝트 루트)를 sys.path에 추가
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 이제 패키지 임포트 가능
from data_module.SWaT import SWaTDataModule, SWaTDataset

In [2]:
from pathlib import Path
import glob

# 1) 프로젝트 루트 추정 (노트북이 data_module/ 안에 있으니 상위 폴더가 루트일 가능성 큼)
PROJECT_ROOT = Path.cwd().parent

# 2) CSV 탐색
patterns = [
    "**/SWaT_Dataset_Normal_v1.csv",
    "**/SWaT_Dataset_Attack_v0.csv",
]
found = {}
for pat in patterns:
    matches = list(PROJECT_ROOT.glob(pat))
    found[pat] = matches

print("== 탐색 결과 ==")
for pat, matches in found.items():
    print(pat)
    for m in matches:
        print("  -", m)

# 3) data_dir 결정 가이드 출력
#    SWaTDataModule는 data_dir/'Swat' 를 붙입니다.
#    즉, normal_csv 경로가  /path/to/.../Swat/SWaT_Dataset_Normal_v1.csv 라면,
#    data_dir는 /path/to/... 가 되어야 함.
normal_paths = found["**/SWaT_Dataset_Normal_v1.csv"]
if normal_paths:
    normal_csv = normal_paths[0]
    # Swat 폴더를 찾는다
    try:
        swat_dir = [p for p in normal_csv.parents if p.name.lower() == "swat"][0]
        suggested_data_dir = swat_dir.parent
        print("\n[제안] summarize_swat_datamodule(data_dir=...)에 넣을 경로:")
        print("data_dir =", suggested_data_dir.as_posix())
        print("\n예시 호출:")
        print(f"_ = summarize_swat_datamodule(data_dir=r'{suggested_data_dir.as_posix()}', "
              "window_size=10, batch_size=64, forecast=False, use_scaler=True)")
    except IndexError:
        print("\n경고: Normal CSV 위에 'Swat' 폴더가 보이지 않습니다.")
        print(" - SWaTDataModule는 data_dir/'Swat' 를 찾습니다.")
        print(" - CSV를 Swat 폴더로 옮기거나, 코드에서 'Swat' 하드코딩을 제거하세요.")
else:
    print("\nNormal CSV를 못 찾았습니다. CSV가 어디에 있는지 확인하세요.")

== 탐색 결과 ==
**/SWaT_Dataset_Normal_v1.csv
  - /home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data/Swat/SWaT_Dataset_Normal_v1.csv
**/SWaT_Dataset_Attack_v0.csv
  - /home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data/Swat/SWaT_Dataset_Attack_v0.csv

[제안] summarize_swat_datamodule(data_dir=...)에 넣을 경로:
data_dir = /home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data

예시 호출:
_ = summarize_swat_datamodule(data_dir=r'/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data', window_size=10, batch_size=64, forecast=False, use_scaler=True)


In [3]:
dm = SWaTDataModule(
    data_dir="/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data",
)
dm.setup()

In [4]:
dm.columns.index('MV303')

21

In [5]:
import math
import pandas as pd

def summarize_swat_datamodule(
    data_dir: str = "data/",
    window_size: int = 10,
    batch_size: int = 64,
    forecast: bool = False,
    use_scaler: bool = True,
) -> pd.DataFrame:
    dm = SWaTDataModule(
        data_dir=data_dir,
        window_size=window_size,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        forecast=forecast,
        use_scaler=use_scaler,
    )
    dm.setup()

    splits = {
        "train": (dm.data_train, dm.train_dataloader()),
        "val":   (dm.data_val,   dm.val_dataloader()),
        "test":  (dm.data_test,  dm.test_dataloader()),
    }

    rows = []
    n_features = len(dm.columns) if dm.columns is not None else None
    for name, (dataset, loader) in splits.items():
        n_samples = len(dataset)                          # 준비되는 윈도우 개수
        bs = loader.batch_size or batch_size
        n_batches = math.ceil(n_samples / bs)

        labels = getattr(dataset, "labels", None)
        n_anom = int(labels.sum()) if labels is not None else None
        n_norm = int((labels == 0).sum()) if labels is not None else None

        timestamps = getattr(dataset, "timestamps", None)
        ts_min = timestamps.min() if timestamps is not None and len(timestamps) > 0 else None
        ts_max = timestamps.max() if timestamps is not None and len(timestamps) > 0 else None

        rows.append({
            "split": name,
            "n_series_windows": n_samples,
            "batch_size": bs,
            "n_batches": n_batches,
            "n_features": n_features,
            "n_norm": n_norm,
            "n_anom": n_anom,
            "ts_start": ts_min,
            "ts_end": ts_max,
            "forecast": forecast,
            "use_scaler": use_scaler,
            "window_size": window_size,
        })

    df = pd.DataFrame(rows)
    display(df)
    return df

# ==== 실행 예시 ====
_ = summarize_swat_datamodule(
    data_dir="/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data",
    window_size=100,
    batch_size=64,
    forecast=False,
    use_scaler=True,
)

,split,n_series_windows,batch_size,n_batches,n_features,n_norm,n_anom,ts_start,ts_end,forecast,use_scaler,window_size
0,train,396000,64,6188,51,396000,0,2015-12-22 16:30:00,2015-12-27 06:29:59,False,True,100
1,val,99000,64,1547,51,99000,0,2015-12-27 06:30:00,2015-12-28 09:59:59,False,True,100
2,test,449919,100,4500,51,395102,54817,2015-12-28 10:00:00,2016-01-02 14:59:59,False,True,100


In [6]:
# ===== PSM 요약 함수 (ipynb용) =====
import math
import pandas as pd

# 필요 시 프로젝트 루트를 sys.path에 추가
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent  # 노트북이 data_module/ 안에 있을 때 가정
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# PSM 모듈 임포트 (패키지/스크립트 혼용 대응)
try:
    from data_module.PSM import PSMDataModule  # 너의 PSMDataModule 이름이 다르면 바꿔라
except Exception as e:
    print("PSMDataModule 임포트 실패:", e)
    raise

def summarize_psm_datamodule(
    data_dir: str,
    window_size: int = 10,
    batch_size: int = 64,
    forecast: bool = False,
    use_scaler: bool = True,
):
    """
    PSMDataModule을 초기화/셋업하고, train/val/test에 대해
    - 준비되는 시계열(윈도우) 개수
    - 배치 개수
    - feature 수
    - 정상/이상 라벨 개수
    - 타임스탬프 시작/끝(있으면)
    를 표로 요약한다.
    """
    dm = PSMDataModule(
        data_dir=data_dir,
        window_size=window_size,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        forecast=forecast,
        use_scaler=use_scaler,
    )
    dm.setup()

    # data_train/data_val/data_test & 대응 dataloader 가정
    splits = {
        "train": (dm.data_train, getattr(dm, "train_dataloader")()),
        "val":   (dm.data_val,   getattr(dm, "val_dataloader")()),
        "test":  (dm.data_test,  getattr(dm, "test_dataloader")()),
    }

    rows = []
    n_features = len(dm.columns) if getattr(dm, "columns", None) is not None else None

    for name, (dataset, loader) in splits.items():
        n_samples = len(dataset)
        bs = getattr(loader, "batch_size", None) or batch_size
        n_batches = math.ceil(n_samples / bs) if bs else None

        labels = getattr(dataset, "labels", None)
        n_anom = int(labels.sum()) if labels is not None else None
        n_norm = int((labels == 0).sum()) if labels is not None else None

        timestamps = getattr(dataset, "timestamps", None)
        ts_min = timestamps.min() if timestamps is not None and len(timestamps) > 0 else None
        ts_max = timestamps.max() if timestamps is not None and len(timestamps) > 0 else None

        rows.append({
            "split": name,
            "n_series_windows": n_samples,
            "batch_size": bs,
            "n_batches": n_batches,
            "n_features": n_features,
            "n_norm": n_norm,
            "n_anom": n_anom,
            "ts_start": ts_min,
            "ts_end": ts_max,
            "forecast": forecast,
            "use_scaler": use_scaler,
            "window_size": window_size,
        })

    df = pd.DataFrame(rows)
    display(df)
    return df

# ==== 실행 예시 ====
_ = summarize_psm_datamodule(
    data_dir="/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data/PSM",  # PSM 폴더 기준 경로
    window_size=100,
    batch_size=64,
    forecast=False,
    use_scaler=True,
)

,split,n_series_windows,batch_size,n_batches,n_features,n_norm,n_anom,ts_start,ts_end,forecast,use_scaler,window_size
0,train,105984,64,1656,None,105984,0,None,None,False,True,100
1,val,26497,64,415,None,26497,0,None,None,False,True,100
2,test,87841,64,1373,None,63460,24381,None,None,False,True,100


In [7]:
# ===== SMD 모든 머신 요약 (ipynb 셀) =====
import sys, math, inspect
from pathlib import Path
import pandas as pd

# 프로젝트 루트 경로를 sys.path에 추가 (노트북이 data_module/ 안에 있을 때 가정)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# SMDDataModule 임포트
try:
    from data_module.SMD import SMDDataModule
except Exception as e:
    raise ImportError(f"SMDDataModule 임포트 실패: {e}")
# ===== SMD 모든 머신 요약 표 (Python 3.8+ 호환) =====
import os
import math
import pandas as pd
from pathlib import Path
from typing import List, Optional

# --- 머신 자동 탐색: data_dir/train/*.txt 파일명에서 machine_id 추출 ---
def discover_smd_machines(data_dir: str) -> List[str]:
    train_dir = Path(data_dir) / "train"
    if not train_dir.exists():
        raise FileNotFoundError(f"'train' 폴더 없음: {train_dir}")
    machines: List[str] = [p.stem for p in sorted(train_dir.glob("*.txt"))]
    if not machines:
        raise FileNotFoundError(f"'train/*.txt'에서 머신을 찾지 못함: {train_dir}")
    return machines

def summarize_smd_all(
    data_dir: str,
    window_size: int = 100,
    batch_size: int = 64,
    forecast: bool = False,
    use_scaler: bool = True,
    machines: Optional[List[str]] = None,   # <-- 수정 포인트
) -> pd.DataFrame:
    # SMDDataModule/SMDataset 가 이미 위에서 정의되어 있다고 가정
    if machines is None:
        machines = discover_smd_machines(data_dir)

    rows = []
    for mid in machines:
        dm = SMDDataModule(
            machine_id=mid,
            data_dir=data_dir,
            window_size=window_size,
            batch_size=batch_size,
            num_workers=0,
            pin_memory=False,
            forecast=forecast,
            use_scaler=use_scaler,
        )
        dm.setup()

        splits = {
            "train": (dm.data_train, dm.train_dataloader()),
            "val":   (dm.data_val,   dm.val_dataloader()),
            "test":  (dm.data_test,  dm.test_dataloader()),
        }

        for split_name, (dataset, loader) in splits.items():
            n_samples = len(dataset)                       # 준비되는 윈도우 개수
            bs = getattr(loader, "batch_size", None) or batch_size
            n_batches = math.ceil(n_samples / bs)

            # feature 수: windows shape = (N, window, F) 가정
            n_features = int(dataset.windows.shape[-1]) if hasattr(dataset, "windows") else None

            labels = getattr(dataset, "labels", None)
            n_anom = int(labels.sum()) if labels is not None else None
            n_norm = int((labels == 0).sum()) if labels is not None else None

            rows.append({
                "machine": mid,
                "split": split_name,
                "n_series_windows": n_samples,
                "batch_size": bs,
                "n_batches": n_batches,
                "n_features": n_features,
                "forecast": forecast,
                "use_scaler": use_scaler,
                "window_size": window_size,
                "n_norm": n_norm,
                "n_anom": n_anom,
            })

    df = pd.DataFrame(rows)
    df["split"] = pd.Categorical(df["split"], categories=["train","val","test"], ordered=True)
    df = df.sort_values(["machine","split"]).reset_index(drop=True)
    display(df)
    return df

# ==== 실행 예시 ====
_ = summarize_smd_all(
    data_dir="/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data/SMD",  # 예: "/home/user/.../data/SMD"
    window_size=100,
    batch_size=64,
    forecast=False,
    use_scaler=True,
)


,machine,split,n_series_windows,batch_size,n_batches,n_features,forecast,use_scaler,window_size,n_norm,n_anom
0,machine-1-1,train,22783,64,356,38,False,True,100,22783,0
1,machine-1-1,val,5696,64,89,38,False,True,100,5696,0
2,machine-1-1,test,28479,64,445,38,False,True,100,25785,2694
3,machine-1-2,train,18955,64,297,38,False,True,100,18955,0
4,machine-1-2,val,4739,64,75,38,False,True,100,4739,0
...,...,...,...,...,...,...,...,...,...,...,...
79,machine-3-8,val,5741,64,90,38,False,True,100,5741,0
80,machine-3-8,test,28704,64,449,38,False,True,100,27333,1371
81,machine-3-9,train,22970,64,359,38,False,True,100,22970,0
82,machine-3-9,val,5743,64,90,38,False,True,100,5743,0


In [3]:
# ===== SMD 모든 머신 요약 표 =====
import os
import sys
import math
import pandas as pd
from pathlib import Path
from typing import List, Optional

# 프로젝트 루트 경로를 sys.path에 추가 (노트북이 data_module/ 안에 있을 때 가정)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# SMDDataModule 임포트
try:
    from data_module.SMD import SMDDataModule
except Exception as e:
    raise ImportError(f"SMDDataModule 임포트 실패: {e}")

# --- 머신 자동 탐색: data_dir/train/*.txt 파일명에서 machine_id 추출 ---
def discover_smd_machines(data_dir: str) -> List[str]:
    train_dir = Path(data_dir) / "train"
    if not train_dir.exists():
        raise FileNotFoundError(f"'train' 폴더 없음: {train_dir}")
    machines: List[str] = [p.stem for p in sorted(train_dir.glob("*.txt"))]
    if not machines:
        raise FileNotFoundError(f"'train/*.txt'에서 머신을 찾지 못함: {train_dir}")
    return machines


def summarize_smd_all(
    data_dir: str,
    window_size: int = 100,
    batch_size: int = 64,
    forecast: bool = False,
    use_scaler: bool = True,
    machines: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    각 machine 별 train/val/test 통계를 만들고,
    마지막에 machine='ALL' 로 전체 합계를 split별로 추가.
    """
    if machines is None:
        machines = discover_smd_machines(data_dir)

    rows = []
    for mid in machines:
        dm = SMDDataModule(
            machine_id=mid,
            data_dir=data_dir,
            window_size=window_size,
            batch_size=batch_size,
            num_workers=0,
            pin_memory=False,
            forecast=forecast,
            use_scaler=use_scaler,
        )
        dm.setup()

        splits = {
            "train": (dm.data_train, dm.train_dataloader()),
            "val":   (dm.data_val,   dm.val_dataloader()),
            "test":  (dm.data_test,  dm.test_dataloader()),
        }

        for split_name, (dataset, loader) in splits.items():
            n_samples = len(dataset)                       # 윈도 개수
            bs = getattr(loader, "batch_size", None) or batch_size
            n_batches = math.ceil(n_samples / bs)

            # feature 수: windows shape = (N, window, F) 가정
            n_features = int(dataset.windows.shape[-1]) if hasattr(dataset, "windows") else None

            labels = getattr(dataset, "labels", None)
            n_anom = int(labels.sum()) if labels is not None else None
            n_norm = int((labels == 0).sum()) if labels is not None else None

            rows.append({
                "machine": mid,
                "split": split_name,
                "n_series_windows": n_samples,
                "batch_size": bs,
                "n_batches": n_batches,
                "n_features": n_features,
                "forecast": forecast,
                "use_scaler": use_scaler,
                "window_size": window_size,
                "n_norm": n_norm,
                "n_anom": n_anom,
            })

    # 상세 데이터프레임
    df = pd.DataFrame(rows)
    df["split"] = pd.Categorical(df["split"], categories=["train", "val", "test"], ordered=True)
    df = df.sort_values(["machine", "split"]).reset_index(drop=True)

    # === 전체 합계(ALL) 추가 ===
    agg = (
        df.groupby("split")
          .agg({
              "n_series_windows": "sum",
              "n_batches":        "sum",
              "n_features":       "first",   # 모든 머신에서 동일하다고 가정
              "n_norm":           "sum",
              "n_anom":           "sum",
              "batch_size":       "first",
              "forecast":         "first",
              "use_scaler":       "first",
              "window_size":      "first",
          })
          .reset_index()
    )
    agg["machine"] = "ALL"
    # 컬럼 순서 맞추기
    agg = agg[df.columns]

    # per-machine + 전체 요약을 한 DF로 합치기
    df_all = pd.concat([df, agg], ignore_index=True)
    df_all = df_all.sort_values(["machine", "split"]).reset_index(drop=True)

    display(df_all)
    return df_all


# ==== 실행 예시 ====
_ = summarize_smd_all(
    data_dir="/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data/SMD",
    window_size=100,
    batch_size=64,
    forecast=False,
    use_scaler=True,
)


/tmp/ipykernel_1009751/3609920489.py:99: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("split")


,machine,split,n_series_windows,batch_size,n_batches,n_features,forecast,use_scaler,window_size,n_norm,n_anom
0,ALL,train,566713,64,8874,38,False,True,100,566713,0
1,ALL,val,141692,64,2234,38,False,True,100,141692,0
2,ALL,test,708420,64,11087,38,False,True,100,678976,29444
3,machine-1-1,train,22783,64,356,38,False,True,100,22783,0
4,machine-1-1,val,5696,64,89,38,False,True,100,5696,0
...,...,...,...,...,...,...,...,...,...,...,...
82,machine-3-8,val,5741,64,90,38,False,True,100,5741,0
83,machine-3-8,test,28704,64,449,38,False,True,100,27333,1371
84,machine-3-9,train,22970,64,359,38,False,True,100,22970,0
85,machine-3-9,val,5743,64,90,38,False,True,100,5743,0


In [1]:
import os
import math
import sys
from pathlib import Path
import pandas as pd

# 프로젝트 루트(노트북이 data_module/ 안에 있다고 가정)
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_module.SMAP_MSL import NASADataModule


def summarize_nasa_datamodule(
    dataset: str,             # "SMAP" 또는 "MSL"
    data_root: str,           # .../SARAD_memory_finch/data 까지
    window_size: int = 100,
    batch_size: int = 64,
    forecast: bool = False,
    use_scaler: bool = True,
) -> pd.DataFrame:
    """
    SMAP / MSL 에 대해 NASADataModule 설정 후
    train/val/test 윈도 개수, 배치 수, feature 수, 정상/이상 개수 등을 요약.
    """

    dataset = dataset.upper()
    assert dataset in ("SMAP", "MSL")

    # 실제 npy 파일이 있는 디렉토리 (지금 구조 기준)
    data_dir = os.path.join(data_root, dataset)

    dm = NASADataModule(
        dataset=dataset,
        data_dir=data_dir,
        window_size=window_size,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        use_scaler=use_scaler,
        forecast=forecast,
    )
    dm.setup()

    splits = {
        "train": (dm.data_train, dm.train_dataloader()),
        "val":   (dm.data_val,   dm.val_dataloader()),
        "test":  (dm.data_test,  dm.test_dataloader()),
    }

    rows = []

    for split_name, (dataset_obj, loader) in splits.items():
        n_samples = len(dataset_obj)
        bs = getattr(loader, "batch_size", None) or batch_size
        n_batches = math.ceil(n_samples / bs)

        # ScalarWindowDataset: windows (N, W, C), targets (N,)
        n_features = int(dataset_obj.windows.shape[-1])
        targets = dataset_obj.targets
        n_anom = int(targets.sum())
        n_norm = int((targets == 0).sum())

        rows.append({
            "dataset": dataset,
            "split": split_name,
            "n_series_windows": n_samples,
            "batch_size": bs,
            "n_batches": n_batches,
            "n_features": n_features,
            "n_norm": n_norm,
            "n_anom": n_anom,
            "forecast": forecast,
            "use_scaler": use_scaler,
            "window_size": window_size,
        })

    df = pd.DataFrame(rows)
    df["split"] = pd.Categorical(df["split"], ["train", "val", "test"], ordered=True)
    df = df.sort_values(["dataset", "split"]).reset_index(drop=True)
    display(df)
    return df


In [2]:
DATA_ROOT = "/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data"

_ = summarize_nasa_datamodule(
    dataset="SMAP",
    data_root=DATA_ROOT,
    window_size=100,
    batch_size=64,
    forecast=False,
    use_scaler=True,
)

_ = summarize_nasa_datamodule(
    dataset="MSL",
    data_root=DATA_ROOT,
    window_size=100,
    batch_size=64,
    forecast=False,
    use_scaler=True,
)


[SMAP] train_raw=(135183, 25), test_raw=(427617, 25), win=100, forecast=False
  -> windows: train=108146, val=27037, test=427617


,dataset,split,n_series_windows,batch_size,n_batches,n_features,n_norm,n_anom,forecast,use_scaler,window_size
0,SMAP,train,108146,64,1690,25,108146,0,False,True,100
1,SMAP,val,27037,64,423,25,27037,0,False,True,100
2,SMAP,test,427617,64,6682,25,372921,54696,False,True,100


[MSL] train_raw=(58317, 55), test_raw=(73729, 55), win=100, forecast=False
  -> windows: train=46653, val=11664, test=73729


,dataset,split,n_series_windows,batch_size,n_batches,n_features,n_norm,n_anom,forecast,use_scaler,window_size
0,MSL,train,46653,64,729,55,46653,0,False,True,100
1,MSL,val,11664,64,183,55,11664,0,False,True,100
2,MSL,test,73729,64,1153,55,65963,7766,False,True,100


In [5]:
import os
import sys
import math
from pathlib import Path

import pandas as pd
from IPython.display import display

# === 공통 설정 ===
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = "/home/user/Desktop/Bongjun/Timeseries_AnomlayDetection/SARAD_memory_finch/data"
WINDOW_SIZE = 100
BATCH_SIZE = 64
FORECAST = False
USE_SCALER = True

# === DataModule 임포트 ===
from data_module.SWaT import SWaTDataModule
from data_module.PSM import PSMDataModule
from data_module.SMD import SMDDataModule
from data_module.SMAP_MSL import NASADataModule   # SMAP / MSL


# ---------- SWaT ----------
def get_swat_stats():
    dm = SWaTDataModule(
        data_dir=DATA_ROOT,
        window_size=WINDOW_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
        forecast=FORECAST,
        use_scaler=USE_SCALER,
    )
    dm.setup()

    n_features = len(dm.columns)
    n_train = len(dm.data_train)
    n_test = len(dm.data_test)

    test_labels = dm.data_test.labels
    n_anom_test = int(test_labels.sum())
    anom_pct = 100.0 * n_anom_test / n_test

    return dict(
        Dataset="SWaT",
        Features=n_features,
        Train=n_train,
        Test=n_test,
        AnomalyPct=anom_pct,
    )


# ---------- PSM ----------
def get_psm_stats():
    dm = PSMDataModule(
        data_dir=os.path.join(DATA_ROOT, "PSM"),
        window_size=WINDOW_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
        forecast=FORECAST,
        use_scaler=USE_SCALER,
    )
    dm.setup()

    # PSMDataModule 에서는 columns 대신 input_size / window 데이터 사용
    n_features = dm.input_size
    n_train = len(dm.data_train)
    n_test = len(dm.data_test)

    test_labels = dm.data_test.labels
    n_anom_test = int(test_labels.sum())
    anom_pct = 100.0 * n_anom_test / n_test

    return dict(
        Dataset="PSM",
        Features=n_features,
        Train=n_train,
        Test=n_test,
        AnomalyPct=anom_pct,
    )


# ---------- SMD (모든 머신 합산) ----------
def discover_smd_machines(data_dir: str):
    train_dir = Path(data_dir) / "train"
    machines = [p.stem for p in sorted(train_dir.glob("*.txt"))]
    return machines


def get_smd_stats():
    data_dir = os.path.join(DATA_ROOT, "SMD")
    machines = discover_smd_machines(data_dir)

    total_train = 0
    total_test = 0
    total_anom_test = 0
    n_features = None

    for mid in machines:
        dm = SMDDataModule(
            machine_id=mid,
            data_dir=data_dir,
            window_size=WINDOW_SIZE,
            batch_size=BATCH_SIZE,
            num_workers=0,
            pin_memory=False,
            forecast=FORECAST,
            use_scaler=USE_SCALER,
        )
        dm.setup()

        total_train += len(dm.data_train)
        total_test += len(dm.data_test)

        test_labels = dm.data_test.labels
        total_anom_test += int(test_labels.sum())

        if n_features is None:
            n_features = int(dm.data_train.windows.shape[-1])

    anom_pct = 100.0 * total_anom_test / total_test

    return dict(
        Dataset="SMD",
        Features=n_features,
        Train=total_train,
        Test=total_test,
        AnomalyPct=anom_pct,
    )


# ---------- SMAP / MSL ----------
def get_nasa_stats(dataset_name: str):
    dataset_name = dataset_name.upper()
    data_dir = os.path.join(DATA_ROOT, dataset_name)

    dm = NASADataModule(
        dataset=dataset_name,
        data_dir=data_dir,
        window_size=WINDOW_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
        use_scaler=USE_SCALER,
        forecast=FORECAST,
    )
    dm.setup()

    train_ds = dm.data_train
    test_ds = dm.data_test

    n_train = len(train_ds)
    n_test = len(test_ds)

    n_features = int(train_ds.windows.shape[-1])

    test_targets = test_ds.targets
    n_anom_test = int(test_targets.sum())
    anom_pct = 100.0 * n_anom_test / n_test

    return dict(
        Dataset=dataset_name,
        Features=n_features,
        Train=n_train,
        Test=n_test,
        AnomalyPct=anom_pct,
    )


# ---------- 전체 5개 데이터셋 요약 및 LaTeX 출력 ----------
rows = []
rows.append(get_swat_stats())
rows.append(get_smd_stats())
rows.append(get_psm_stats())
rows.append(get_nasa_stats("MSL"))
rows.append(get_nasa_stats("SMAP"))

df = pd.DataFrame(rows)
display(df)

print("\n% LaTeX table rows")
for r in rows:
    print(
        f"{r['Dataset']} & "
        f"{r['Features']} & "
        f"{r['Train']:,} & "
        f"{r['Test']:,} & "
        f"{r['AnomalyPct']:.2f} \\\\"
    )


[MSL] train_raw=(58317, 55), test_raw=(73729, 55), win=100, forecast=False
  -> windows: train=46653, val=11664, test=73729
[SMAP] train_raw=(135183, 25), test_raw=(427617, 25), win=100, forecast=False
  -> windows: train=108146, val=27037, test=427617


,Dataset,Features,Train,Test,AnomalyPct
0,SWaT,51,396000,449919,12.183749
1,SMD,38,566713,708420,4.156291
2,PSM,25,105984,87841,27.755832
3,MSL,55,46653,73729,10.533169
4,SMAP,25,108146,427617,12.790885



% LaTeX table rows
SWaT & 51 & 396,000 & 449,919 & 12.18 \\
SMD & 38 & 566,713 & 708,420 & 4.16 \\
PSM & 25 & 105,984 & 87,841 & 27.76 \\
MSL & 55 & 46,653 & 73,729 & 10.53 \\
SMAP & 25 & 108,146 & 427,617 & 12.79 \\
